# Metricas que se van a usar para la evalución de los modelos basados en léxicos

## VADER
Vader al ser un modelo basado en valencias, lo normal sería usar métricas continuas como el MAE, MSE, RMSE, R^2 y MAPE si contamos las estrellas como un rango continuo [0, 5]. Además de esto, podríamos utilizar las 5 estrellas como 5 categorías y utilizar Cohens Kappa para ver también como clasífica este modelo realizando una aproximación de la valencia a la categoría más cercana. También podriamos dividir las estrellas en una clasificación binaria, con dos clases que serían positivas (2,5 a 5 estrellas) o negativas (0 a 2,5 estrellas) y ver también como se comportaría el modelo en clasificación binaria (accuracy, precision, F1-SCORE...)

## AFINN
Lo mismo que VADER ya que se comporta igual

## NRC
NRC es bastante diferente, ya que los valores más importantes que da este modelo son los sentiminetos asociados a cada reseña, cosa que no se puede evaluar cuantitativamente ya que no tenemos un valor objetivo. Lo suyo sería realizar un análisis no tanto de precisión sino cualitativo, para ver como se asocian las emociones con los valores de las estrellas o palabras específicas.


In [9]:
import polars as p

In [22]:
csv_scored = '../results/yelp_academic_dataset_review_scored.csv'

def regression_evaluation_metrics(csv_with_scores):
    
    df = pl.scan_csv(csv_with_scores)
    
    metricas_vader = df.select([
        (pl.col("stars") - pl.col("vader_score")).abs().mean().alias("MAE_vader"),
        (pl.col("stars") - pl.col("vader_score")).pow(2).mean().alias("MSE_vader"),
        (pl.col("stars") - pl.col("vader_score")).pow(2).mean().sqrt().alias("RMSE_vader"),
        (1 - (pl.col("stars") - pl.col("vader_score")).pow(2).sum() / 
         (pl.col("stars") - pl.col("stars").mean()).pow(2).sum()).alias("R2_vader"),
        pl.corr("stars", "vader_score", method="spearman").alias("Spearman_vader")
    ])
    
    metricas_afinn = df.select([
        (pl.col("stars") - pl.col("afinn_score")).abs().mean().alias("MAE_afinn"),
        (pl.col("stars") - pl.col("afinn_score")).pow(2).mean().alias("MSE_afinn"),
        (pl.col("stars") - pl.col("afinn_score")).pow(2).mean().sqrt().alias("RMSE_afinn"),
        (1 - (pl.col("stars") - pl.col("vader_score")).pow(2).sum() / 
         (pl.col("stars") - pl.col("stars").mean()).pow(2).sum()).alias("R2_afinn"),
        pl.corr("stars", "afinn_score", method="spearman").alias("Spearman_afinn")
    ])   
    
    print("Métricas de regresión de VADER:")
    print(metricas_vader.collect())

    print("Métricas de regresión de AFINN:")
    print(metricas_afinn.collect())

regression_evaluation_metrics(csv_scored)

Métricas de regresión de VADER:
shape: (1, 5)
┌───────────┬───────────┬────────────┬──────────┬────────────────┐
│ MAE_vader ┆ MSE_vader ┆ RMSE_vader ┆ R2_vader ┆ Spearman_vader │
│ ---       ┆ ---       ┆ ---        ┆ ---      ┆ ---            │
│ f64       ┆ f64       ┆ f64        ┆ f64      ┆ f64            │
╞═══════════╪═══════════╪════════════╪══════════╪════════════════╡
│ 0.813329  ┆ 1.521367  ┆ 1.233437   ┆ 0.304221 ┆ 0.512293       │
└───────────┴───────────┴────────────┴──────────┴────────────────┘
Métricas de regresión de AFINN:
shape: (1, 5)
┌───────────┬───────────┬────────────┬──────────┬────────────────┐
│ MAE_afinn ┆ MSE_afinn ┆ RMSE_afinn ┆ R2_afinn ┆ Spearman_afinn │
│ ---       ┆ ---       ┆ ---        ┆ ---      ┆ ---            │
│ f64       ┆ f64       ┆ f64        ┆ f64      ┆ f64            │
╞═══════════╪═══════════╪════════════╪══════════╪════════════════╡
│ 0.848419  ┆ 1.628902  ┆ 1.276284   ┆ 0.304221 ┆ 0.482755       │
└───────────┴───────────┴────────────